In [9]:
import pandas as pd
import numpy as np

from pandas.api.types import is_numeric_dtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, StandardScaler
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
TARGET_COLUMN = "label"

train_users = pd.read_csv("data/train_users.csv")
news_articles = pd.read_csv("data/news_articles.csv")

train_users['label'].unique()

<StringArray>
['user_3', 'user_2', 'user_1']
Length: 3, dtype: str

In [10]:
print("Train Users Shape:", train_users.shape)
print("News Articles Shape:", news_articles.shape)


Train Users Shape: (2000, 33)
News Articles Shape: (209527, 6)


In [11]:
# Data Preprocessing
train_users = train_users.drop(columns=["user_id"])

for col in train_users.columns:
    if is_numeric_dtype(train_users[col]):
        train_users[col] = train_users[col].fillna(train_users[col].median())
    else:
        train_users[col] = train_users[col].fillna(train_users[col].mode()[0])

X = train_users.drop(columns=[TARGET_COLUMN])
y = train_users[TARGET_COLUMN]

categorical_cols = X.select_dtypes(include=["object", "string"]).columns

ordinal_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X[categorical_cols] = ordinal_encoder.fit_transform(X[categorical_cols])
train_users[categorical_cols] = ordinal_encoder.transform(train_users[categorical_cols])

target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

print("Mapping:")
for cls, idx in zip(target_encoder.classes_, range(len(target_encoder.classes_))):
    print(f"{cls} -> {idx}")

Mapping:
user_1 -> 0
user_2 -> 1
user_3 -> 2


In [12]:
#Data Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [13]:
# Train - Test split
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

In [14]:
#training classifier
classifier = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

classifier.fit(X_train, y_train)

print('Model Trained')


Model Trained


In [15]:
# model evaluation
y_pred = classifier.predict(X_val)

print('Classification Report: ')

print(
    classification_report(
        y_val,
        y_pred,
        target_names=target_encoder.classes_
    )
)

Classification Report: 
              precision    recall  f1-score   support

      user_1       0.89      0.86      0.87       142
      user_2       0.97      0.88      0.92       142
      user_3       0.84      0.97      0.90       116

    accuracy                           0.90       400
   macro avg       0.90      0.90      0.90       400
weighted avg       0.90      0.90      0.90       400

